# COMP9517 - Computer Vision - Group Project Demo

## Setting up the Environment

In [ ]:
%%bash

if command -v uv 2> /dev/null
then
    echo Running uv sync...
    uv sync
    uv pip install git+https://github.com/lucasb-eyer/pydensecrf.git
    uv pip install "pyzmq<26" --break-system-packages
else
    echo "Please install uv to setup the environment"
    echo "Hint: Run 'pip install uv'"
fi

printf "\nRunning in Directory: %s" "$(pwd)"


/home/linuxbrew/.linuxbrew/bin/uv
Running uv sync...


Resolved 54 packages in 0.84ms
Uninstalled 28 packages in 34ms
Installed 1 package in 5ms
 - asttokens==3.0.1
 - comm==0.2.3
 - debugpy==1.8.20
 - decorator==5.2.1
 - executing==2.2.1
 - ipykernel==7.2.0
 - ipython==9.12.0
 - ipython-pygments-lexers==1.1.1
 - jedi==0.19.2
 - jupyter-client==8.8.0
 - jupyter-core==5.9.1
 - matplotlib-inline==0.2.1
 - nest-asyncio==1.6.0
 - packaging==26.1
 + packaging==26.0
 - parso==0.8.6
 - pexpect==4.9.0
 - platformdirs==4.9.6
 - prompt-toolkit==3.0.52
 - psutil==7.2.2
 - ptyprocess==0.7.0
 - pure-eval==0.2.3
 - pydensecrf==1.0 (from git+https://github.com/lucasb-eyer/pydensecrf.git@2723c7fa4f2ead16ae1ce3d8afe977724bb8f87f)
 - pygments==2.20.0
 - pyzmq==27.1.0
 - stack-data==0.6.3
 - tornado==6.5.5
 - traitlets==5.14.3
 - wcwidth==0.6.0
Using Python 3.12.3 environment at: /home/tj_devil/Workspace/unsw/cs9517/cs9517_group_project/.venv
Resolved 1 package in 827ms
Installed 1 package in 3ms
 + pydensecrf==1.0 (from git+https://github.com/lucasb-eyer/py


Running in Directory: /home/tj_devil/Workspace/unsw/cs9517/cs9517_group_project/notebooks

## Classic CV methods

In [9]:
%%bash
unset MPLBACKEND
source ../.venv/bin/activate

jq -r 'keys[]' ../configs/traditional_cv.json | while read -r method; do
    # Run and Evaluate traditional cv algorithms
    cmd="../scripts/classic_cv.py -R $method -C traditional_cv.json"
    echo "Running Command $cmd"
    eval "$cmd"

    # Plot cross-level comparison
    cmd="../scripts/compare.py -m robustness -D test -R $method"
    echo "Running Command $cmd"
    eval "$cmd"
done

# Compare all traditional cv methods
cmd="../scripts/compare.py -m cross -D test --method cv -C traditional_cv.json"
echo Running Command "$cmd"
eval "$cmd"

Running Command ../scripts/classic_cv.py -R crf_method -C traditional_cv.json
Running Command ../scripts/compare.py -m robustness -D test -R crf_method
Running Command ../scripts/classic_cv.py -R edge_method -C traditional_cv.json
Running Command ../scripts/compare.py -m robustness -D test -R edge_method
Running Command ../scripts/classic_cv.py -R excessive_green_method -C traditional_cv.json
Running Command ../scripts/compare.py -m robustness -D test -R excessive_green_method
Running Command ../scripts/classic_cv.py -R grabcut_method -C traditional_cv.json
Running Command ../scripts/compare.py -m robustness -D test -R grabcut_method
Running Command ../scripts/classic_cv.py -R hsv_segmentation_method -C traditional_cv.json
Running Command ../scripts/compare.py -m robustness -D test -R hsv_segmentation_method
Running Command ../scripts/classic_cv.py -R kmeans_method -C traditional_cv.json
Running Command ../scripts/compare.py -m robustness -D test -R kmeans_method
Running Command ../scr

## Machine Learning methods

In [10]:
%%bash
unset MPLBACKEND
source ../.venv/bin/activate

capitalize() {
    tr '[:lower:]' '[:upper:]'
}

channel_modes=("rgb" "hsv" "exg")
unset mode

for m in "${channel_modes[@]}"; do
    mode="$mode""$m"
    run_name=$(echo "$mode" | capitalize)

    # Train random forest model
    cmd="python3 -m project.machine_learning.train_rf -R RF_$run_name -fm $mode"
    echo Running Command "$cmd"
    eval "$cmd"

    # Evaluate random forest model
    cmd="python3 -m project.machine_learning.evaluate_rf -R RF_$run_name -m test -fm $mode"
    echo Running Command "$cmd"
    eval "$cmd"

    # Train logistic regression model
    cmd="python3 -m project.machine_learning.train_lr -R LR_$run_name -fm $mode"
    echo Running Command "$cmd"
    eval "$cmd"

    # Evaluate logistic regression model
    cmd="python3 -m project.machine_learning.evaluate_lr -R LR_$run_name -m test -fm $mode"
    echo Running Command "$cmd"
    eval "$cmd"

    # Concat channels
    mode="$mode"_
done

unset mode

# Compare between classifiers
cmd="../scripts/plot_classifier_comparison.py"
echo Running Command "$cmd"
eval "$cmd"

# Compare within the random forests
cmd="../scripts/plot_rf_feature_comparison.py"
echo Running Command "$cmd"
eval "$cmd"

Running Command python3 -m project.machine_learning.train_rf -R RF_RGB -fm rgb
[OK] Trained RF run=RF_RGB
Train time: 2.44s
Validation inference time: 1.92s
Validation accuracy: 0.8755592367209022
Running Command python3 -m project.machine_learning.evaluate_rf -R RF_RGB -m test -fm rgb
[OK] Evaluated RF run=RF_RGB, mode=test
Inference time: 1.95s
Accuracy: 0.8649159160855716
Running Command python3 -m project.machine_learning.train_lr -R LR_RGB -fm rgb
[OK] Trained LR run=LR_RGB
Train time: 0.18s
Validation inference time: 0.02s
Validation accuracy: 0.8794873450413223
Running Command python3 -m project.machine_learning.evaluate_lr -R LR_RGB -m test -fm rgb
[OK] Evaluated LR run=LR_RGB, mode=test
Inference time: 0.02s
Accuracy: 0.8684270349087465
Running Command python3 -m project.machine_learning.train_rf -R RF_RGB_HSV -fm rgb_hsv
[OK] Trained RF run=RF_RGB_HSV
Train time: 4.22s
Validation inference time: 1.88s
Validation accuracy: 0.874414197012741
Running Command python3 -m project.m

## Deep learning methods

In [11]:
%%bash
unset MPLBACKEND
source ../.venv/bin/activate

# resume=last.pt

jq -r 'keys[]' ../configs/networks.json | while read -r run; do
    # Train the neural network
    cmd="python3 -m project.deep_learning.train_neural_network -R $run -C networks.json"

    if [ -n "$resume" ]; then
        cmd="$cmd -r $resume"
    fi

    echo "Running Command $cmd"
    eval "$cmd"

    # Evaluate the neural network
    cmd="python3 -m project.deep_learning.evaluate_neural_network -R $run -C networks.json -m test"
    echo "Running Command $cmd"
    eval "$cmd"
    # Compare between robustness levels
    cmd="../scripts/compare.py -m robustness -D test -R $run"
    echo "Running Command $cmd"
    eval "$cmd"
done

# Compare all deep learning models
cmd="../scripts/compare.py -m cross -D test --method dl -C networks.json"
echo Running Command "$cmd"
eval "$cmd"

Running Command python3 -m project.deep_learning.train_neural_network -R ASPPResUnet -C networks.json


2026-04-15 21:11:28,217 [INFO] train logger: epoch 1 | train loss: 0.384720 | val loss: 0.224267
2026-04-15 21:11:50,731 [INFO] train logger: epoch 2 | train loss: 0.239282 | val loss: 0.183972
2026-04-15 21:11:53,129 [INFO] train logger: epoch 3 | train loss: 0.202912 | val loss: 0.173287
2026-04-15 21:12:03,794 [INFO] train logger: epoch 4 | train loss: 0.192069 | val loss: 0.162294
2026-04-15 21:12:14,220 [INFO] train logger: epoch 5 | train loss: 0.188786 | val loss: 0.157988
2026-04-15 21:12:24,811 [INFO] train logger: epoch 6 | train loss: 0.176223 | val loss: 0.220782
2026-04-15 21:12:34,595 [INFO] train logger: epoch 7 | train loss: 0.178777 | val loss: 0.165090
2026-04-15 21:12:44,693 [INFO] train logger: epoch 8 | train loss: 0.166606 | val loss: 0.157085
2026-04-15 21:12:55,156 [INFO] train logger: epoch 9 | train loss: 0.167052 | val loss: 0.156767
2026-04-15 21:13:05,760 [INFO] train logger: epoch 10 | train loss: 0.160161 | val loss: 0.154672
2026-04-15 21:13:25,890 [INFO

Running Command python3 -m project.deep_learning.evaluate_neural_network -R ASPPResUnet -C networks.json -m test
Running Command ../scripts/compare.py -m robustness -D test -R ASPPResUnet
Running Command python3 -m project.deep_learning.train_neural_network -R AttentionASPPResUnet -C networks.json


2026-04-15 21:18:53,444 [INFO] train logger: epoch 1 | train loss: 0.395332 | val loss: 0.221156
2026-04-15 21:19:05,214 [INFO] train logger: epoch 2 | train loss: 0.237468 | val loss: 0.181753
2026-04-15 21:19:16,719 [INFO] train logger: epoch 3 | train loss: 0.200723 | val loss: 0.174009
2026-04-15 21:19:28,226 [INFO] train logger: epoch 4 | train loss: 0.190110 | val loss: 0.162579
2026-04-15 21:19:39,904 [INFO] train logger: epoch 5 | train loss: 0.187276 | val loss: 0.160051
2026-04-15 21:19:51,409 [INFO] train logger: epoch 6 | train loss: 0.174367 | val loss: 0.239493
2026-04-15 21:20:11,690 [INFO] train logger: epoch 7 | train loss: 0.176929 | val loss: 0.173419
2026-04-15 21:20:12,887 [INFO] train logger: epoch 8 | train loss: 0.167375 | val loss: 0.164405
2026-04-15 21:20:23,647 [INFO] train logger: epoch 9 | train loss: 0.166696 | val loss: 0.155433
2026-04-15 21:20:35,281 [INFO] train logger: epoch 10 | train loss: 0.160300 | val loss: 0.153320
2026-04-15 21:20:46,856 [INFO

Running Command python3 -m project.deep_learning.evaluate_neural_network -R AttentionASPPResUnet -C networks.json -m test
Running Command ../scripts/compare.py -m robustness -D test -R AttentionASPPResUnet
Running Command python3 -m project.deep_learning.train_neural_network -R ResUnet -C networks.json


2026-04-15 21:28:11,954 [INFO] train logger: epoch 1 | train loss: 0.422217 | val loss: 0.246123
2026-04-15 21:28:20,467 [INFO] train logger: epoch 2 | train loss: 0.244942 | val loss: 0.186494
2026-04-15 21:28:28,996 [INFO] train logger: epoch 3 | train loss: 0.202024 | val loss: 0.171430
2026-04-15 21:28:37,375 [INFO] train logger: epoch 4 | train loss: 0.188854 | val loss: 0.161695
2026-04-15 21:28:45,808 [INFO] train logger: epoch 5 | train loss: 0.186165 | val loss: 0.156684
2026-04-15 21:28:54,318 [INFO] train logger: epoch 6 | train loss: 0.173001 | val loss: 0.222335
2026-04-15 21:29:02,483 [INFO] train logger: epoch 7 | train loss: 0.174927 | val loss: 0.166268
2026-04-15 21:29:10,606 [INFO] train logger: epoch 8 | train loss: 0.164319 | val loss: 0.160133
2026-04-15 21:29:18,688 [INFO] train logger: epoch 9 | train loss: 0.164463 | val loss: 0.153770
2026-04-15 21:29:27,239 [INFO] train logger: epoch 10 | train loss: 0.158880 | val loss: 0.156118
2026-04-15 21:29:35,403 [INFO

Running Command python3 -m project.deep_learning.evaluate_neural_network -R ResUnet -C networks.json -m test
Running Command ../scripts/compare.py -m robustness -D test -R ResUnet
Running Command python3 -m project.deep_learning.train_neural_network -R UNet_Baseline -C networks.json


2026-04-15 21:35:20,078 [INFO] train logger: epoch 1 | train loss: 0.672276 | val loss: 0.639058
2026-04-15 21:35:26,814 [INFO] train logger: epoch 2 | train loss: 0.629572 | val loss: 0.555823
2026-04-15 21:35:33,452 [INFO] train logger: epoch 3 | train loss: 0.470901 | val loss: 0.402529
2026-04-15 21:35:40,078 [INFO] train logger: epoch 4 | train loss: 0.315289 | val loss: 0.222811
2026-04-15 21:35:46,685 [INFO] train logger: epoch 5 | train loss: 0.250015 | val loss: 0.221205
2026-04-15 21:35:53,384 [INFO] train logger: epoch 6 | train loss: 0.252678 | val loss: 0.205114
2026-04-15 21:36:00,088 [INFO] train logger: epoch 7 | train loss: 0.245755 | val loss: 0.198010
2026-04-15 21:36:06,680 [INFO] train logger: epoch 8 | train loss: 0.231565 | val loss: 0.188159
2026-04-15 21:36:13,380 [INFO] train logger: epoch 9 | train loss: 0.232294 | val loss: 0.198238
2026-04-15 21:36:19,631 [INFO] train logger: epoch 10 | train loss: 0.223033 | val loss: 0.209078
2026-04-15 21:36:26,114 [INFO

Running Command python3 -m project.deep_learning.evaluate_neural_network -R UNet_Baseline -C networks.json -m test
Running Command ../scripts/compare.py -m robustness -D test -R UNet_Baseline
Running Command ../scripts/compare.py -m cross -D test --method dl -C networks.json
